# Silver snap counts

## Pipeline overview

**Purpose:** Preserve nflverse/PFR snap-count records as clean, source-shaped Silver data for all positions and game types. Snap counts support later participation analysis, but this notebook does not map PFR identities to GSIS or create player-week features.

| Role | Files | Columns | Grain |
| --- | --- | ---: | --- |
| Data input | `data/bronze/snap_counts/snap_counts_{season}.parquet`, 2023–2026 | 16 per season | PFR player-game-team |
| Validation only | `data/silver/schedules/silver_schedules_{season}.parquet` | 46 per season | `game_id` |
| Output | `data/silver/snap_counts/silver_snap_counts_{season}.parquet` | 16 per season | `game_id + pfr_player_id + team` |

**Cleaning:** Remove exact full-row duplicates; fail on conflicting source keys; normalize text fields and three whole-valued snap counts. Preserve the source percentages, position, PFR identifiers, and all source rows, including zero-snap rows.

**Blocking checks:** Source schema and season; non-null, unique grain; safe snap-count conversion; nonnegative snaps and percentages in 0–1; consistent zero-snap/zero-percent behavior; schedule context; PFR-game mapping; source-value reconciliation; and Parquet round trip.

**Diagnostics, not repairs:** Rows with zero snaps in all units and rows with positive snaps but zero source percentage. No source percentage is recalculated. Current-season row totals are not hard-coded because 2026 is in progress.


## 1. Set paths and type contract

PFR player and game IDs remain strings. Source snap counts arrive as whole-valued floats and become nullable integers only after exact-conversion checks.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SEASONS = [2023, 2024, 2025, 2026]
EXPECTED_COLUMNS = 16
SNAP_KEY = ["game_id", "pfr_player_id", "team"]
TEXT_COLUMNS = [
    "game_id", "pfr_game_id", "game_type", "player",
    "pfr_player_id", "position", "team", "opponent",
]
SNAP_COLUMNS = ["offense_snaps", "defense_snaps", "st_snaps"]
PERCENT_COLUMNS = ["offense_pct", "defense_pct", "st_pct"]

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "ROADMAP.md").exists():
    assert PROJECT_ROOT != PROJECT_ROOT.parent, "Open this notebook inside the project checkout"
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze" / "snap_counts"
SCHEDULE_DIR = PROJECT_ROOT / "data" / "silver" / "schedules"
OUTPUT_DIR = PROJECT_ROOT / "data" / "silver" / "snap_counts"


## 2. Inspect source schemas and load each season

Each Bronze partition must contain the same 16 named fields in source order. Schedules are required only to validate game context.


In [2]:
source_columns = None
bronze_partitions = {}
source_inventory = []

for season in SEASONS:
    source_path = BRONZE_DIR / f"snap_counts_{season}.parquet"
    schedule_path = SCHEDULE_DIR / f"silver_schedules_{season}.parquet"
    assert source_path.exists(), f"Missing Bronze snap-count input: {source_path}"
    assert schedule_path.exists(), f"Missing schedule validation input: {schedule_path}"

    column_names = pq.read_schema(source_path).names
    assert len(column_names) == EXPECTED_COLUMNS, (season, len(column_names))
    assert set(TEXT_COLUMNS + SNAP_COLUMNS + PERCENT_COLUMNS + ["season", "week"]) == set(column_names)
    if source_columns is None:
        source_columns = column_names
    else:
        assert column_names == source_columns, f"Schema drift in {source_path}"

    bronze = pd.read_parquet(source_path)
    assert bronze.columns.tolist() == source_columns
    assert bronze["season"].eq(season).all(), f"Unexpected season in {source_path}"
    bronze_partitions[season] = bronze
    source_inventory.append({
        "season": season, "bronze_rows": len(bronze),
        "columns": len(bronze.columns), "source": str(source_path.relative_to(PROJECT_ROOT)),
    })

pd.DataFrame(source_inventory)


,season,bronze_rows,columns,source
0,2023,26540,16,data/bronze/snap_counts/snap_counts_2023.parquet
1,2024,26615,16,data/bronze/snap_counts/snap_counts_2024.parquet
2,2025,26613,16,data/bronze/snap_counts/snap_counts_2025.parquet
3,2026,7370,16,data/bronze/snap_counts/snap_counts_2026.parquet


## 3. Remove exact duplicates and validate the source key

Only identical full rows may be removed. Different records with the same PFR player-game-team key must stop the build.


In [3]:
deduplicated_partitions = {}
duplicate_counts = {}

for season in SEASONS:
    bronze = bronze_partitions[season]
    exact_duplicates = bronze.duplicated(keep="first")
    duplicate_counts[season] = int(exact_duplicates.sum())
    source = bronze.loc[~exact_duplicates].reset_index(drop=True)
    assert source[SNAP_KEY].notna().all().all(), season
    assert not source.duplicated(SNAP_KEY).any(), (
        season, source.loc[source.duplicated(SNAP_KEY, keep=False), SNAP_KEY].head(10)
    )
    assert len(source) == len(bronze) - duplicate_counts[season]
    deduplicated_partitions[season] = source

pd.DataFrame([
    {"season": season, "exact_duplicates_removed": duplicate_counts[season],
     "rows_after_deduplication": len(deduplicated_partitions[season])}
    for season in SEASONS
])


,season,exact_duplicates_removed,rows_after_deduplication
0,2023,0,26540
1,2024,0,26615
2,2025,0,26613
3,2026,0,7370


## 4. Normalize types and reconcile every source value

Percentage fields are already source fractions; the notebook validates but does not recompute or round them. Whole-number conversion must preserve both values and nulls.


In [4]:
silver_partitions = {}

for season in SEASONS:
    source = deduplicated_partitions[season]
    silver = source.copy()

    for column in TEXT_COLUMNS:
        silver[column] = silver[column].astype("string")
    for column in SNAP_COLUMNS:
        numeric_values = pd.to_numeric(silver[column], errors="raise")
        assert numeric_values.notna().all(), (season, column, "missing snap count")
        assert np.isfinite(numeric_values).all(), (season, column, "non-finite")
        assert numeric_values.mod(1).eq(0).all(), (season, column, "fractional")
        assert numeric_values.abs().le(2**53).all(), (season, column, "unsafe float integer")
        silver[column] = numeric_values.astype("Int64")

    assert silver.columns.tolist() == source_columns
    assert len(silver) == len(source)
    unchanged_columns = [
        column for column in source_columns
        if column not in TEXT_COLUMNS + SNAP_COLUMNS
    ]
    pd.testing.assert_frame_equal(silver[unchanged_columns], source[unchanged_columns])
    for column in TEXT_COLUMNS:
        pd.testing.assert_series_equal(
            silver[column], source[column].astype("string"), check_names=False
        )
    for column in SNAP_COLUMNS:
        pd.testing.assert_series_equal(
            silver[column].astype("Float64"),
            pd.to_numeric(source[column]).astype("Float64"),
            check_names=False,
        )
    silver_partitions[season] = silver

print("All 16 source columns and values reconciled after type normalization.")


All 16 source columns and values reconciled after type normalization.


## 5. Validate snap values and scheduled games

Zero snaps require zero percentage. A positive snap count with zero source percentage is reported later, not overwritten. Schedules validate context but supply no output columns.


In [5]:
all_snaps = pd.concat(silver_partitions.values(), ignore_index=True)
assert not all_snaps.duplicated(SNAP_KEY).any()
assert all_snaps[["game_id", "pfr_game_id", "season", "week", "game_type", "player", "pfr_player_id", "position", "team", "opponent"]].notna().all().all()
assert all_snaps["week"].between(1, 23).all()
assert all_snaps["game_type"].isin(["REG", "WC", "DIV", "CON", "SB"]).all()
assert all_snaps.groupby("game_id")["pfr_game_id"].nunique().eq(1).all()
assert all_snaps.groupby("pfr_game_id")["game_id"].nunique().eq(1).all()

for snaps_column, pct_column in zip(SNAP_COLUMNS, PERCENT_COLUMNS):
    snaps = all_snaps[snaps_column]
    pct = all_snaps[pct_column]
    assert snaps.ge(0).all(), snaps_column
    assert pct.notna().all(), pct_column
    assert np.isfinite(pct.to_numpy()).all(), pct_column
    assert pct.between(0, 1).all(), pct_column
    assert not (snaps.eq(0) & pct.gt(0)).any(), (snaps_column, pct_column)

schedule_checks = []
for season in SEASONS:
    schedules = pd.read_parquet(
        SCHEDULE_DIR / f"silver_schedules_{season}.parquet",
        columns=["game_id", "season", "week", "game_type", "home_team", "away_team"],
    )
    assert schedules["game_id"].notna().all()
    assert not schedules.duplicated("game_id").any()

    snap_context = silver_partitions[season][["game_id", "season", "week", "game_type", "team", "opponent"]].drop_duplicates()
    checked = snap_context.merge(
        schedules, on="game_id", how="left",
        suffixes=("", "_schedule"), validate="many_to_one", indicator=True,
    )
    assert checked["_merge"].eq("both").all(), season
    assert checked["season"].eq(checked["season_schedule"]).all(), season
    assert checked["week"].eq(checked["week_schedule"]).all(), season
    assert checked["game_type"].eq(checked["game_type_schedule"]).all(), season
    valid_opponents = (
        checked["team"].eq(checked["home_team"]) & checked["opponent"].eq(checked["away_team"])
    ) | (
        checked["team"].eq(checked["away_team"]) & checked["opponent"].eq(checked["home_team"])
    )
    assert valid_opponents.all(), season
    schedule_checks.append({"season": season, "matched_snap_team_games": len(checked)})

pd.DataFrame(schedule_checks)


,season,matched_snap_team_games
0,2023,570
1,2024,570
2,2025,570
3,2026,158


## 6. Review source exceptions without filtering

Six current source rows have zero snaps in every unit. They are retained but are not positive snap-based participation evidence. A source percentage can also disagree with positive snaps; that is shown rather than repaired.


In [6]:
diagnostics = []
for season in SEASONS:
    silver = silver_partitions[season]
    diagnostics.append({
        "season": season,
        "zero_all_snap_rows": int(silver[SNAP_COLUMNS].eq(0).all(axis=1).sum()),
        "offense_positive_snap_zero_pct": int((silver["offense_snaps"].gt(0) & silver["offense_pct"].eq(0)).sum()),
        "defense_positive_snap_zero_pct": int((silver["defense_snaps"].gt(0) & silver["defense_pct"].eq(0)).sum()),
        "st_positive_snap_zero_pct": int((silver["st_snaps"].gt(0) & silver["st_pct"].eq(0)).sum()),
    })

diagnostic_summary = pd.DataFrame(diagnostics)
diagnostic_summary


,season,zero_all_snap_rows,offense_positive_snap_zero_pct,defense_positive_snap_zero_pct,st_positive_snap_zero_pct
0,2023,2,0,0,0
1,2024,1,0,0,0
2,2025,2,0,0,1
3,2026,1,0,0,0


## 7. Persist and verify each Silver partition

Every in-memory partition has passed validation before the first write. Reloaded Parquet must reproduce all 16 fields, values, nulls, and normalized types.


In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
validation_rows = []

for season in SEASONS:
    silver = silver_partitions[season]
    output_path = OUTPUT_DIR / f"silver_snap_counts_{season}.parquet"
    silver.to_parquet(output_path, index=False)
    reloaded = pd.read_parquet(output_path)
    pd.testing.assert_frame_equal(reloaded, silver)
    assert reloaded.columns.tolist() == source_columns
    assert len(reloaded.columns) == EXPECTED_COLUMNS
    assert not reloaded.duplicated(SNAP_KEY).any()
    validation_rows.append({
        "season": season,
        "bronze_rows": len(bronze_partitions[season]),
        "exact_duplicates_removed": duplicate_counts[season],
        "silver_rows": len(reloaded),
        "columns": len(reloaded.columns),
        "output": str(output_path.relative_to(PROJECT_ROOT)),
    })

validation_summary = pd.DataFrame(validation_rows)
assert validation_summary["silver_rows"].eq(
    validation_summary["bronze_rows"] - validation_summary["exact_duplicates_removed"]
).all()
validation_summary


,season,bronze_rows,exact_duplicates_removed,silver_rows,columns,output
0,2023,26540,0,26540,16,data/silver/snap_counts/silver_snap_counts_202...
1,2024,26615,0,26615,16,data/silver/snap_counts/silver_snap_counts_202...
2,2025,26613,0,26613,16,data/silver/snap_counts/silver_snap_counts_202...
3,2026,7370,0,7370,16,data/silver/snap_counts/silver_snap_counts_202...
